# No double-booking, step by step

A hands-on lab. Run one cell at a time, read what happened, then move on. You will create a GiST exclusion constraint to prevent overlapping settlements on the same account, then prove it works by attempting a conflicting insert.

Each step shows the real interface: SQL where the step is a query, and Python for control operations.

**Before you start:** finish `00-setup/` so `LB_PROFILE`, `LB_PROJECT`, `LB_BRANCH`, `LB_SCHEMA` are set and the bank is loaded on your branch.

## Setup, connect to your branch

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", "common", "scripts"))
from lakebase import connect, run_sql

SCHEMA = os.environ.get("LB_SCHEMA", "bank")
print("connected; schema:", SCHEMA)

## Step 1, inspect the settlement table

See the structure of settlements (clearing windows on accounts). The SQL:

```sql
SELECT settlement_id, account_id, starts_at, ends_at FROM bank.settlement LIMIT 3
```

In [ ]:
sql = "SELECT settlement_id, account_id, starts_at, ends_at FROM bank.settlement LIMIT 3"
rows = run_sql(sql)
print("Sample settlements:")
for row in rows:
    print(f"  {row}")

## Step 2, create GiST exclusion constraint

Enable the btree_gist extension, then add an exclusion constraint that prevents overlapping time ranges for the same account. The SQL:

```sql
CREATE EXTENSION IF NOT EXISTS btree_gist;
ALTER TABLE bank.settlement
  ADD CONSTRAINT IF NOT EXISTS no_account_overlap
  EXCLUDE USING gist (account_id WITH =,
                      tstzrange(starts_at, ends_at) WITH &&)
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        try:
            cur.execute("CREATE EXTENSION IF NOT EXISTS btree_gist")
            print("btree_gist extension created")
        except:
            pass

        # Postgres has no ADD CONSTRAINT IF NOT EXISTS; drop then add so it's idempotent.
        cur.execute("ALTER TABLE bank.settlement DROP CONSTRAINT IF EXISTS no_account_overlap")
        sql = """
        ALTER TABLE bank.settlement
          ADD CONSTRAINT no_account_overlap
          EXCLUDE USING gist (account_id WITH =,
                              tstzrange(starts_at, ends_at) WITH &&)
        """
        try:
            cur.execute(sql)
            print("Exclusion constraint created: no_account_overlap")
        except Exception as e:
            print(f"Could not create constraint: {e}")

## Step 3, attempt an overlapping insert

Try to insert a settlement that overlaps with an existing one on the same account. The constraint should reject it.

In [ ]:
import datetime

# Get an existing settlement (and its trade, to satisfy the FK) to overlap.
stls = run_sql("""
SELECT settlement_id, trade_id, account_id, starts_at, ends_at
  FROM bank.settlement
 WHERE account_id IS NOT NULL AND starts_at IS NOT NULL AND ends_at IS NOT NULL
 LIMIT 1
""")

if not stls:
    print("No existing settlements found")
else:
    settlement_id, trade_id, account_id, starts_at, ends_at = stls[0]

    # Try to insert an overlapping settlement
    with connect() as conn:
        with conn.cursor() as cur:
            # Clean any prior test row so this is re-runnable
            cur.execute("DELETE FROM bank.settlement WHERE settlement_id = 'STL-OVERLAP'")
            try:
                # Create a settlement that overlaps with the existing one
                overlap_start = starts_at
                overlap_end = ends_at + datetime.timedelta(hours=1)

                cur.execute("""
                INSERT INTO bank.settlement
                  (settlement_id, trade_id, account_id, seq, status, starts_at, ends_at)
                VALUES
                  (%s, %s, %s, %s, %s, %s, %s)
                """, ('STL-OVERLAP', trade_id, account_id, 99, 'pending', overlap_start, overlap_end))
                print("ERROR: Overlap should have been rejected!")
            except Exception as e:
                error_msg = str(e)
                if "no_account_overlap" in error_msg or "conflicting key" in error_msg or "exclusion" in error_msg.lower():
                    print("Good: the database rejected the overlapping settlement.")
                    print(f"  {error_msg.splitlines()[0][:120]}")
                else:
                    print(f"Unexpected error: {error_msg[:120]}")

## Step 4, EXPLAIN shows constraint check

Run EXPLAIN to see the constraint check in the query plan.

In [ ]:
sql = ("EXPLAIN INSERT INTO bank.settlement "
       "(settlement_id, trade_id, account_id, seq, status, starts_at, ends_at) "
       "SELECT 'stl-explain', trade_id, 'CTA-100240', 98, 'pending', now(), now() + interval '1 hour' "
       "FROM bank.settlement LIMIT 1")

with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(sql)
        rows = cur.fetchall()
        print("Query plan (the exclusion check runs when the row is written, so it isn't listed here):")
        for row in rows:
            print(f"  {row[0]}")

## What you learned

You created a GiST exclusion constraint that prevents overlapping time ranges for the same account. This constraint is checked at the database level, not the application level, so no double-booking is possible.

Next: **Watch the planner** uses EXPLAIN ANALYZE to understand when indexes matter.